In [ ]:
import os
import shutil
import re
import math
import random
from pathlib import Path
from collections import defaultdict
import copy
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import transforms

#global variables
to_tensor = transforms.ToTensor()
IMG_SIZE = 320

CLASS_NAMES = ["grade0", "grade1", "grade2", "grade3"]
NA = 3

STRIDES = [4, 8, 16]

ANCHORS_ORIGINALS = [
    [[5, 7],    [8, 15],   [17, 12]],
    [[15, 31],  [31, 23],   [30, 60]],
    [[58, 45],  [78, 99],   [187, 163]]
]
GRID_SIZES = [(IMG_SIZE // s, IMG_SIZE // s) for s in STRIDES]

ANCHORS = [torch.tensor(a, dtype=torch.float32) for a in ANCHORS_ORIGINALS]

#label extraction
def get_id(filename):
    filename = filename.split('.')[0]
    return filename.split('_')[1]

def burn_grade(mask_name):
    mask_name = mask_name.split('.')[0]
    cls = int(mask_name.split('_')[3][1])
    return cls

def get_box(mask):
    mask_binary = mask != 0
    if mask.max() == 0:
        return None
    ys, xs = np.where(mask_binary)
    x_min, x_max = int(xs.min()), int(xs.max())
    y_min, y_max = int(ys.min()), int(ys.max())
    return (x_min, y_min, x_max + 1, y_max + 1)

def to_yolo(xmin, ymin, xmax, ymax):
    w = xmax - xmin
    h = ymax - ymin
    xc = (xmin + xmax) / 2.0 / 320
    yc = (ymin + ymax) / 2.0 / 320
    w = w * 1.1
    h = h * 1.1
    wn = w / 320
    hn = h / 320
    if wn > 1: wn = 1
    if hn > 1: hn = 1
    return xc, yc, wn, hn

#augmentation functions
def elastic_deform(img, alpha=40, sigma=6):
    height, width = img.shape[:2]
    dx = np.random.rand(height, width) * 2 - 1
    dy = np.random.rand(height, width) * 2 - 1
    dx = cv2.GaussianBlur(dx, (0, 0), sigma) * alpha
    dy = cv2.GaussianBlur(dy, (0, 0), sigma) * alpha
    x, y = np.meshgrid(np.arange(width), np.arange(height))
    map_x = (x + dx).astype(np.float32)
    map_y = (y + dy).astype(np.float32)
    return cv2.remap(img, map_x, map_y,
                     interpolation=cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_REFLECT)

def flip(img):
    return cv2.flip(img, 1)

def adjustcontrast(img, alpha, beta):
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def adjustlight(img, alpha, beta):
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def gaussian_blur(img):
    sigma = 1.3
    return cv2.GaussianBlur(img, (5, 5), sigma)

def add_noise(img, sigma=10):
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)



#folder maker
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists("/content/poze.zip"):
    shutil.copy("/content/drive/MyDrive/fire/poze.zip", "/content/poze.zip")
    shutil.unpack_archive("/content/poze.zip", "/content/poze_2")
if not os.path.exists("/content/masks.zip"):
    shutil.copy("/content/drive/MyDrive/fire/masks.zip", "/content/masks.zip")
    shutil.unpack_archive("/content/masks.zip", "/content/masks_2")

print("images:", len(os.listdir("/content/poze_2/poze")))
print("masks: ", len(os.listdir("/content/masks_2/masks")))

#finding the initial nr of labels
TARGET = 3000
samples = []
univ_counts = [0, 0, 0, 0]
IMG_DIR = r"/content/poze_2/poze"
MASK_DIR = r"/content/masks_2/masks"

def extract_labels():
    for img in sorted(os.listdir(IMG_DIR)):
        counts = [0, 0, 0, 0]
        labels_tmp = []
        img_id = get_id(img)
        for mask in sorted(os.listdir(MASK_DIR)):
            if get_id(mask) != img_id:
                continue
            mask_img = cv2.imread(os.path.join(MASK_DIR, mask),
                                  cv2.IMREAD_GRAYSCALE)
            box = get_box(mask_img)
            if box is None:
                continue
            grade = burn_grade(mask)
            counts[grade] += 1
            labels_tmp.append(mask)
        samples.append([img, labels_tmp, counts])
        for i in range(len(counts)):
            univ_counts[i] += counts[i]
    return samples, univ_counts

samples, univ_counts = extract_labels()
print(univ_counts)



EVAL_FRAC = 0.2
MIN_EVAL_IMAGES_PER_GRADE = 2
SPLIT_SEED = 42

def split_samples(samples, eval_frac=EVAL_FRAC,
                  min_per_grade=MIN_EVAL_IMAGES_PER_GRADE, seed=SPLIT_SEED):
    uniq = list({s[0]: s for s in samples}.values())
    random.Random(seed).shuffle(uniq)
    n_eval = int(eval_frac * len(uniq))
    eval_idx = set()


    n_with = [sum(1 for s in uniq if s[2][g] > 0) for g in range(4)]
    for c in sorted(range(4), key=lambda g: n_with[g]):
        have = sum(1 for i in eval_idx if uniq[i][2][c] > 0)
        for i, s in enumerate(uniq):
            if have >= min_per_grade:
                break
            if i not in eval_idx and s[2][c] > 0:
                eval_idx.add(i)
                have += 1


    for i in range(len(uniq)):
        if len(eval_idx) >= n_eval:
            break
        eval_idx.add(i)

    train = [s for i, s in enumerate(uniq) if i not in eval_idx]
    ev = [s for i, s in enumerate(uniq) if i in eval_idx]

    print(f"split: {len(train)} train images, {len(ev)} eval images")
    for c in range(4):
        n_ev = sum(1 for s in ev if s[2][c] > 0)
        n_tr = sum(1 for s in train if s[2][c] > 0)
        print(f"  grade{c}: images with it -> train {n_tr}, eval {n_ev}")
        if n_ev < min_per_grade:
            print(f"  WARNING: only {n_ev} eval image(s) for grade{c} (not enough images in the dataset)")
        if n_tr == 0:
            print(f"  WARNING: no train image contains grade{c}")
    return train, ev

train_samples, eval_samples = split_samples(samples)


univ_counts = [sum(s[2][c] for s in train_samples) for c in range(4)]
print("train counts before augmentation:", univ_counts)







POZE_OUT = "/content/poze_2_aug"
MASK_OUT = "/content/masks_2_aug"
EVAL_POZE_OUT = "/content/poze_2_eval"
EVAL_MASK_OUT = "/content/masks_2_eval"
MASK_DIR = r"/content/masks_2/masks"
IMG_DIR = r"/content/poze_2/poze"

def augm(img_name, img, folder, k):
    augment_vect = [0, 0, 0, 0, 0, 0, 0, 0]
    for i in range(k):
        aug = img.copy()
        add = ''
        img_name2 = img_name.split('.')
        ext = img_name2[1]
        if augment_vect[0] == 1:
            aug = elastic_deform(aug)
            add += '0'
        if augment_vect[1] == 1:
            aug = flip(aug)
            add += '1'
        if augment_vect[2] == 1:
            if ext == 'jpg':
                aug = adjustcontrast(aug, alpha=1.1, beta=0)
            add += '2'
        if augment_vect[3] == 1:
            if ext == 'jpg':
                aug = gaussian_blur(aug)
            add += '3'
        if augment_vect[4] == 1:
            if ext == 'jpg':
                aug = add_noise(aug)
            add += '4'
        if augment_vect[5] == 1:
            if ext == 'jpg':
                aug = adjustlight(aug, alpha=1, beta=15)
            add += '5'
        if augment_vect[6] == 1:
            if ext == 'jpg':
                aug = adjustcontrast(aug, alpha=1.1, beta=0)
            add += '6'
        if augment_vect[7] == 1:
            if ext == 'jpg':
                aug = adjustlight(aug, alpha=1, beta=15)
            add += '7'

        img_name2 = img_name2[0].split('_')
        img_name2[1] = img_name2[1] + add
        img_name2 = '_'.join(img_name2)
        ok = cv2.imwrite(os.path.join(folder, img_name2 + '.' + ext), aug)
        if not ok:
            print("FAILURE:", os.path.join(folder, img_name2 + '.' + ext))

        augment_vect[0] += 1
        if augment_vect[0] == 2:
            augment_vect[0] = 0
            augment_vect[1] += 1
        if augment_vect[1] == 2:
            augment_vect[1] = 0
            augment_vect[2] += 1
        if augment_vect[2] == 2:
            augment_vect[2] = 0
            augment_vect[3] += 1
        if augment_vect[3] == 2:
            augment_vect[3] = 0
            augment_vect[4] += 1
        if augment_vect[4] == 2:
            augment_vect[4] = 0
            augment_vect[5] += 1
        if augment_vect[5] == 2:
            augment_vect[5] = 0
            augment_vect[6] += 1
        if augment_vect[6] == 2:
            augment_vect[6] = 0
            augment_vect[7] += 1

def augment(sample):
    global univ_counts
    img_name = sample[0]
    labels = sample[1]
    counts = sample[2]
    k = 256

    for i in range(len(counts)):
        while k > 0 and univ_counts[i] + counts[i] * k > TARGET:
            k -= 1

    if k == 0: return None
    for j in range(len(counts)):
        univ_counts[j] += (k - 1) * counts[j]
    img = cv2.imread(os.path.join(IMG_DIR, img_name))
    augm(img_name, img, POZE_OUT, k)
    for i in range(len(labels)):
        mask_name = labels[i]
        mask = cv2.imread(os.path.join(MASK_DIR, mask_name))
        augm(mask_name, mask, MASK_OUT, k)


for d in (POZE_OUT, MASK_OUT, EVAL_POZE_OUT, EVAL_MASK_OUT):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)


for s in eval_samples:
    shutil.copy2(os.path.join(IMG_DIR, s[0]), os.path.join(EVAL_POZE_OUT, s[0]))
    for mask_name in s[1]:
        shutil.copy2(os.path.join(MASK_DIR, mask_name), os.path.join(EVAL_MASK_OUT, mask_name))


for idx in range(len(train_samples)):
    print(idx, univ_counts)
    augment(train_samples[idx])

print(univ_counts)




class Conv(nn.Module):
    def __init__(self, in_ch, out_ch, k, s, p):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.silu(self.bn(self.conv(x)))

class Bottleneck(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.cv1 = Conv(channels, channels, 1, 1, 0)
        self.cv2 = Conv(channels, channels, 3, 1, 1)

    def forward(self, x):
        x1 = self.cv1(x)
        x2 = self.cv2(x1)
        return x + x2

class C3(nn.Module):
    def __init__(self, n , in_ch, out_ch=None ):
        super().__init__()
        out_ch = out_ch or in_ch
        self.n = n
        hidden = out_ch // 2
        self.cv1 = Conv(in_ch, hidden, 1, 1, 0)
        self.cv2 = Conv(in_ch, hidden, 1, 1, 0)
        self.cv3 = Conv(out_ch, out_ch, 1, 1, 0)
        self.btlnck = Bottleneck(hidden)

    def forward(self, x):
        y1 = self.btlnck(self.cv1(x))
        y2 = self.cv2(x)
        for i in range(self.n):
            y1 = self.btlnck(y1)
        return self.cv3(torch.cat([y1, y2], dim=1))

class SPPF(nn.Module):
    def __init__(self, in_ch, pool_size=5):
        super().__init__()
        hidden = in_ch // 4
        self.cv1 = Conv(in_ch, hidden, 3, 1, 1)
        self.cv2 = Conv(hidden * 4, in_ch, 3, 1, 1)
        self.pool = nn.MaxPool2d(pool_size, stride=1, padding=pool_size // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.cv1 = Conv(3, 64, 6, 2, 2)
        self.cv2 = Conv(64, 128, 3, 1, 1)
        self.c3_1 = C3(1,128, 128)
        self.cv3 = Conv(128, 256, 3, 2, 1)
        self.c3_2 = C3(3,256, 256)
        self.cv4 = Conv(256, 512, 3, 2, 1)
        self.c3_3 = C3(3,512, 512)
        self.cv5 = Conv(512, 1024, 3, 2, 1)
        self.c3_4 = C3(3,1024, 1024)
        self.sppf = SPPF(1024)

    def forward(self, x):
        x = self.cv1(x)
        x = self.c3_1(self.cv2(x))
        p3 = self.c3_2(self.cv3(x))
        p4 = self.c3_3(self.cv4(p3))
        p5 = self.sppf(self.c3_4(self.cv5(p4)))
        return p3, p4, p5

class PANet(nn.Module):
    def __init__(self, num_classes=3, anchors_per_scale=3):
        super().__init__()
        out_ch = anchors_per_scale * (num_classes + 5)
        c3, c4, c5 = 256, 512, 1024
        self.lat_p5 = Conv(c5, c4, 3, 1, 1)
        self.up = nn.Upsample(None, scale_factor=2, mode="nearest")
        self.fuse_p4 = C3(1,c4 * 2, c4)
        self.lat_p4 = Conv(c4, c3, 3, 1, 1)
        self.up2 = nn.Upsample(None, scale_factor=2, mode="nearest")
        self.fuse_p3 = C3(1,c3 * 2, c3)
        self.down_p3 = Conv(c3, c3, 3, 2, 1)
        self.fuse_n4 = C3(1,c3 * 2, c4)
        self.down_n4 = Conv(c4, c4, 3, 2, 1)
        self.fuse_n5 = C3(1,c4 * 2, c5)
        self.head_s = nn.Conv2d(c3, out_ch, 1, 1, 0)
        self.head_m = nn.Conv2d(c4, out_ch, 1, 1, 0)
        self.head_l = nn.Conv2d(c5, out_ch, 1, 1, 0)

    def forward(self, p3, p4, p5):
        p5r = self.lat_p5(p5)
        p4f = self.fuse_p4(torch.cat([self.up(p5r), p4], dim=1))
        p4r = self.lat_p4(p4f)
        n3 = self.fuse_p3(torch.cat([self.up(p4r), p3], dim=1))
        n4 = self.fuse_n4(torch.cat([self.down_p3(n3), p4r], dim=1))
        n5 = self.fuse_n5(torch.cat([self.down_n4(n4), p5r], dim=1))
        return self.head_s(n3), self.head_m(n4), self.head_l(n5)

class DetectionHead(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.nc = num_classes
        self.no = num_classes + 5
        self.na = 3
        self.strides = STRIDES
        self.anchors = ANCHORS

    def forward(self, feature_maps):
        p3, p4, p5 = feature_maps
        out_p3 = self._decode(p3, 0)
        out_p4 = self._decode(p4, 1)
        out_p5 = self._decode(p5, 2)
        return torch.cat([out_p3, out_p4, out_p5], dim=1)

    def _decode(self, raw, scale_idx):
        B, _, H, W = raw.shape
        stride = self.strides[scale_idx]
        anchors = self.anchors[scale_idx].to(raw.device).float()
        raw = raw.view(B, self.na, self.no, H, W).permute(0, 1, 3, 4, 2).contiguous()
        pred = torch.sigmoid(raw)
        gy, gx = torch.meshgrid(
            torch.arange(H, device=raw.device),
            torch.arange(W, device=raw.device),
            indexing='ij')
        gx = gx.view(1, 1, H, W).float()
        gy = gy.view(1, 1, H, W).float()
        bx = (2.0 * pred[..., 0] - 0.5 + gx) * stride
        by = (2.0 * pred[..., 1] - 0.5 + gy) * stride
        anc_w = anchors[:, 0].view(1, self.na, 1, 1)
        anc_h = anchors[:, 1].view(1, self.na, 1, 1)
        bw = (2.0 * pred[..., 2]) ** 2 * anc_w
        bh = (2.0 * pred[..., 3]) ** 2 * anc_h
        out = torch.cat([
            torch.stack([bx, by, bw, bh], dim=-1),
            pred[..., 4:]
        ], dim=-1)
        return out.view(B, self.na * H * W, self.no)

class YOLOv5(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = Backbone()
        self.neck = PANet(num_classes=num_classes)
        self.head = DetectionHead(num_classes=num_classes)

    def forward_raw(self, x):
        p3, p4, p5 = self.backbone(x)
        return self.neck(p3, p4, p5)

    def forward(self, x):
        return self.head(self.forward_raw(x))

#making the database

def make_data(img_folder, mask_folder):
    image_dict = {}
    for img in sorted(os.listdir(img_folder)):
        image_dict.update({get_id(img): img})
    mask_dict = {}
    for mask in sorted(os.listdir(mask_folder)):
        mask_dict.setdefault(get_id(mask), []).append(mask)
    return image_dict, mask_dict

train_img_dict, train_mask_dict = make_data(POZE_OUT, MASK_OUT)
eval_img_dict, eval_mask_dict = make_data(EVAL_POZE_OUT, EVAL_MASK_OUT)

class Database(Dataset):

    def __init__(self, img_dict, mask_dict, img_dir, mask_dir):
        self.img_dict = img_dict
        self.mask_dict = mask_dict
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.ids = list(img_dict.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]
        img_name = self.img_dict[img_id]
        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 320), interpolation=cv2.INTER_LINEAR)
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        masks = []

        for mask_name in self.mask_dict.get(img_id, []):
            mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (320, 320), interpolation=cv2.INTER_NEAREST)
            bbox = get_box(mask)
            if bbox is None:
                continue
            xmin, ymin, xmax, ymax = bbox
            x, y, w, h = to_yolo(xmin, ymin, xmax, ymax)
            grade = burn_grade(mask_name)
            masks.append([x, y, w, h, grade])
        if masks:
            masks = torch.tensor(masks, dtype=torch.float32)
        else:
            masks = torch.zeros((0, 5), dtype=torch.float32)
        return img, masks


train_set = Database(train_img_dict, train_mask_dict, POZE_OUT, MASK_OUT)
eval_set = Database(eval_img_dict, eval_mask_dict, EVAL_POZE_OUT, EVAL_MASK_OUT)

print(f"train: {len(train_set)} (augmented)  eval: {len(eval_set)} (originals, not augmented)")

def collate_fn(batch):
    imgs = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return imgs, targets

train_loader = DataLoader(train_set, batch_size=8, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
eval_loader  = DataLoader(eval_set,  batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)


EVAL_SAVE_PATH = "/content/drive/MyDrive/fire/model-1"


#loss functions
def bbox_giou(pred_boxes, target_boxes, eps=1e-7):
    p_x1 = pred_boxes[..., 0] - pred_boxes[..., 2] / 2
    p_y1 = pred_boxes[..., 1] - pred_boxes[..., 3] / 2
    p_x2 = pred_boxes[..., 0] + pred_boxes[..., 2] / 2
    p_y2 = pred_boxes[..., 1] + pred_boxes[..., 3] / 2
    t_x1 = target_boxes[..., 0] - target_boxes[..., 2] / 2
    t_y1 = target_boxes[..., 1] - target_boxes[..., 3] / 2
    t_x2 = target_boxes[..., 0] + target_boxes[..., 2] / 2
    t_y2 = target_boxes[..., 1] + target_boxes[..., 3] / 2
    p_area = (p_x2 - p_x1).clamp(min=0) * (p_y2 - p_y1).clamp(min=0)
    t_area = (t_x2 - t_x1).clamp(min=0) * (t_y2 - t_y1).clamp(min=0)
    inter_x1 = torch.max(p_x1, t_x1)
    inter_y1 = torch.max(p_y1, t_y1)
    inter_x2 = torch.min(p_x2, t_x2)
    inter_y2 = torch.min(p_y2, t_y2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = p_area + t_area - inter + eps
    iou = inter / union
    c_x1 = torch.min(p_x1, t_x1)
    c_y1 = torch.min(p_y1, t_y1)
    c_x2 = torch.max(p_x2, t_x2)
    c_y2 = torch.max(p_y2, t_y2)
    c_area = (c_x2 - c_x1).clamp(min=0) * (c_y2 - c_y1).clamp(min=0) + eps
    return iou - (c_area - union) / c_area

def giou_loss(pred_boxes, target_boxes):
    return (1.0 - bbox_giou(pred_boxes, target_boxes)).mean()

def wh_iou(wh1, wh2):
    wh1 = wh1[:, None, :]
    wh2 = wh2[None, :, :]
    inter = torch.min(wh1, wh2).prod(dim=2)
    union = wh1.prod(dim=2) + wh2.prod(dim=2) - inter
    return inter / (union + 1e-7)

def assign_targets(tgt, anchors_per_scale, strides, grid_sizes, na):
    matched = []
    scale_offsets = []
    running = 0
    for (H, W) in grid_sizes:
        scale_offsets.append(running)
        running += na * H * W
    for n in range(tgt.shape[0]):
        gx, gy, gw, gh = tgt[n, 0], tgt[n, 1], tgt[n, 2], tgt[n, 3]
        gt_wh = torch.tensor([[gw, gh]], device=tgt.device)
        best_iou, best_scale, best_anchor = -1.0, 0, 0
        for s in range(len(strides)):
            ious = wh_iou(gt_wh, anchors_per_scale[s].to(tgt.device))[0]
            a = int(torch.argmax(ious))
            if ious[a] > best_iou:
                best_iou = float(ious[a])
                best_scale = s
                best_anchor = a
        H, W = grid_sizes[best_scale]
        stride = strides[best_scale]
        raw_col = int(gx / stride)
        raw_row = int(gy / stride)
        col = min(max(raw_col, 0), W - 1)
        row = min(max(raw_row, 0), H - 1)
        local = best_anchor * (H * W) + row * W + col
        global_idx = scale_offsets[best_scale] + local
        matched.append(global_idx)
    return matched

def raw_to_flat(raw_maps, na, no):
    flats = []
    for raw in raw_maps:
        B, _, H, W = raw.shape
        x = raw.view(B, na, no, H, W).permute(0, 1, 3, 4, 2).contiguous()
        flats.append(x.view(B, na * H * W, no))
    return torch.cat(flats, dim=1)

def decode_boxes_flat(raw_maps, strides, anchors_units, na, no):
    boxes = []
    for s, raw in enumerate(raw_maps):
        B, _, H, W = raw.shape
        stride = strides[s]
        anc = anchors_units[s].to(raw.device).float()
        x = raw.view(B, na, no, H, W).permute(0, 1, 3, 4, 2).contiguous()
        p = torch.sigmoid(x[..., 0:4])
        gy, gx = torch.meshgrid(
            torch.arange(H, device=raw.device),
            torch.arange(W, device=raw.device),
            indexing='ij')
        gx = gx.view(1, 1, H, W).float()
        gy = gy.view(1, 1, H, W).float()
        bx = (2.0 * p[..., 0] - 0.5 + gx) * stride
        by = (2.0 * p[..., 1] - 0.5 + gy) * stride
        anc_w = anc[:, 0].view(1, na, 1, 1)
        anc_h = anc[:, 1].view(1, na, 1, 1)
        bw = (2.0 * p[..., 2]) ** 2 * anc_w * stride
        bh = (2.0 * p[..., 3]) ** 2 * anc_h * stride
        xywh = torch.stack([bx, by, bw, bh], dim=-1)
        boxes.append(xywh.view(B, na * H * W, 4))
    return torch.cat(boxes, dim=1)

def compute_detection_loss(head_out, targets,
                           anchors_px, strides, grid_sizes, na,
                           num_classes,
                           img_size=320,
                           box_w=1, obj_w=2, cls_w=1):
    device = head_out.device
    eps = 1e-7
    total_box = torch.zeros(1, device=device)
    total_obj = torch.zeros(1, device=device)
    total_cls = torch.zeros(1, device=device)
    B = head_out.shape[0]
    for b in range(B):
        pred = head_out[b]
        pred_box = pred[:, 0:4]
        obj_prob = pred[:, 4].clamp(eps, 1 - eps)
        cls_prob = pred[:, 5:].clamp(eps, 1 - eps)
        tgt = targets[b].to(device)
        obj_target = torch.zeros(pred.shape[0], device=device)
        if tgt.shape[0] > 0:
            tgt_px = tgt.clone()
            tgt_px[:, 0:4] = tgt_px[:, 0:4] * img_size
            matched_pred_idx = assign_targets(tgt_px, anchors_px, strides, grid_sizes, na)
            total_box += giou_loss(pred_box[matched_pred_idx], tgt_px[:, 0:4])
            matched_cls = cls_prob[matched_pred_idx]
            grades = tgt_px[:, 4].long()
            cls_onehot = torch.zeros_like(matched_cls)
            cls_onehot[torch.arange(grades.shape[0], device=device), grades] = 1.0
            class_weights = torch.tensor([1., 3., 2., 1.], device=device)
            bce_cls = nn.functional.binary_cross_entropy(
                matched_cls, cls_onehot, reduction='none')
            bce_cls = bce_cls * class_weights
            total_cls += bce_cls.mean()
            obj_target[matched_pred_idx] = 1.0
        obj_loss_elem = nn.functional.binary_cross_entropy(
            obj_prob, obj_target, reduction='none')
        pos_mask = obj_target > 0.5
        n_pos = pos_mask.sum().clamp(min=1)
        n_neg = (~pos_mask).sum().clamp(min=1)
        pos_weight = n_neg.float() / n_pos.float()
        weights = torch.ones_like(obj_target)
        weights[pos_mask] = pos_weight
        total_obj += (obj_loss_elem * weights).sum() / weights.sum()
    loss = box_w * total_box + obj_w * total_obj + cls_w * total_cls
    return loss / B, (total_box.item(), total_obj.item(), total_cls.item())

#NMS functions
def box_giou(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2[None, :] - inter + 1e-7
    iou = inter / union
    c_lt = torch.min(boxes1[:, None, :2], boxes2[None, :, :2])
    c_rb = torch.max(boxes1[:, None, 2:], boxes2[None, :, 2:])
    c_wh = (c_rb - c_lt).clamp(min=0)
    c_area = c_wh[:, :, 0] * c_wh[:, :, 1] + 1e-7
    return iou - (c_area - union) / c_area

def box_iou(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2[None, :] - inter + 1e-7
    return inter / union

def non_max_suppression(pred, conf_thresh=0.25, iou_thresh=0.45, nc=4):
    obj = pred[:, 4]
    keep = obj > conf_thresh
    pred = pred[keep]
    if pred.shape[0] == 0:
        return torch.zeros((0, 6), device=pred.device)
    obj = pred[:, 4]
    cls_scores = pred[:, 5:]
    cls_conf, cls_id = cls_scores.max(dim=1)
    score = obj * cls_conf
    keep = score > conf_thresh
    pred, score, cls_id = pred[keep], score[keep], cls_id[keep]
    if pred.shape[0] == 0:
        return torch.zeros((0, 6), device=pred.device)
    cx, cy, w, h = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    boxes = torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=1)
    final = []
    order = score.argsort(descending=True)
    while order.numel() > 0:
        i = order[0]
        final.append(torch.cat([boxes[i], score[i:i+1], cls_id[i].float().view(1)]))
        if order.numel() == 1:
            break
        ious = box_iou(boxes[i:i+1], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_thresh]
    if not final:
        return torch.zeros((0, 6), device=pred.device)
    return torch.stack(final)

def xywh_to_xyxy(boxes):
    x, y, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    return torch.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dim=1)

def head_to_nms_input(head_out):
    return head_out

#metrics print functions
def compute_ap(recall, precision):
    mrec = torch.cat([torch.tensor([0.0]), recall, torch.tensor([1.0])])
    mpre = torch.cat([torch.tensor([0.0]), precision, torch.tensor([0.0])])
    for i in range(mpre.numel() - 1, 0, -1):
        mpre[i - 1] = torch.max(mpre[i - 1], mpre[i])
    idx = torch.where(mrec[1:] != mrec[:-1])[0]
    return torch.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]).item()

def evaluate_map(all_detections, all_ground_truths, num_classes, iou_thresh=0.5):
    aps = {}
    for c in range(num_classes):
        scores_list, tp_list, fp_list = [], [], []
        n_gt = 0
        for dets, gts in zip(all_detections, all_ground_truths):
            dets_c = dets[dets[:, 5] == c] if dets.numel() else dets.new_zeros((0, 6))
            gts_c = gts[gts[:, 4] == c] if gts.numel() else gts.new_zeros((0, 5))
            n_gt += gts_c.shape[0]
            if dets_c.shape[0] == 0:
                continue
            order = dets_c[:, 4].argsort(descending=True)
            dets_c = dets_c[order]
            matched = torch.zeros(gts_c.shape[0], dtype=torch.bool)
            for d in range(dets_c.shape[0]):
                scores_list.append(dets_c[d, 4].item())
                if gts_c.shape[0] == 0:
                    tp_list.append(0); fp_list.append(1)
                    continue
                ious = box_iou(dets_c[d:d + 1, :4], gts_c[:, :4])[0]
                best_iou, best_g = ious.max(0)
                if best_iou >= iou_thresh and not matched[best_g]:
                    tp_list.append(1); fp_list.append(0)
                    matched[best_g] = True
                else:
                    tp_list.append(0); fp_list.append(1)
        if n_gt == 0:
            aps[c] = float('nan')
            continue
        if len(scores_list) == 0:
            aps[c] = 0.0
            continue
        scores = torch.tensor(scores_list)
        tp = torch.tensor(tp_list, dtype=torch.float32)
        fp = torch.tensor(fp_list, dtype=torch.float32)
        order = scores.argsort(descending=True)
        tp, fp = tp[order], fp[order]
        tp_cum = torch.cumsum(tp, dim=0)
        fp_cum = torch.cumsum(fp, dim=0)
        recall = tp_cum / (n_gt + 1e-7)
        precision = tp_cum / (tp_cum + fp_cum + 1e-7)
        aps[c] = compute_ap(recall, precision)
    valid = [v for v in aps.values() if v == v]
    mAP = sum(valid) / len(valid) if valid else 0.0
    return aps, mAP

def build_confusion_matrix(all_detections, all_ground_truths, num_classes, iou_thresh=0.5):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for dets, gts in zip(all_detections, all_ground_truths):
        if dets.numel() == 0 or gts.numel() == 0:
            continue
        order = dets[:, 4].argsort(descending=True)
        dets = dets[order]
        matched = torch.zeros(gts.shape[0], dtype=torch.bool)
        for d in range(dets.shape[0]):
            pred_cls = int(dets[d, 5])
            ious = box_iou(dets[d:d + 1, :4], gts[:, :4])[0]
            best_iou, best_g = ious.max(0)
            if best_iou >= iou_thresh and not matched[best_g]:
                gt_cls = int(gts[best_g, 4])
                cm[pred_cls, gt_cls] += 1
                matched[best_g] = True
    return cm

def evaluate_mean_giou(all_detections, all_ground_truths, num_classes=4,
                       iou_thresh=0.5, img_size=IMG_SIZE):
    giou_sum = [0.0] * num_classes
    giou_cnt = [0] * num_classes
    for dets, gts in zip(all_detections, all_ground_truths):
        if gts.numel() == 0 or dets.numel() == 0:
            continue
        order = dets[:, 4].argsort(descending=True)
        dets = dets[order]
        matched = torch.zeros(gts.shape[0], dtype=torch.bool)
        for d in range(dets.shape[0]):
            ious = box_iou(dets[d:d + 1, :4], gts[:, :4])[0]
            best_iou, best_g = ious.max(0)
            if best_iou >= iou_thresh and not matched[best_g]:
                matched[best_g] = True
                giou = box_giou(dets[d:d + 1, :4], gts[best_g:best_g + 1, :4])[0, 0]
                gt_cls = int(gts[best_g, 4])
                giou_sum[gt_cls] += giou.item()
                giou_cnt[gt_cls] += 1
    per_class = [
        (giou_sum[c] / giou_cnt[c]) if giou_cnt[c] > 0 else float('nan')
        for c in range(num_classes)
    ]
    return per_class, giou_cnt

def plot_eval_loss(eval_losses):
    epochs = range(1, len(eval_losses) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, eval_losses, marker='o', color='tab:red')
    plt.xlabel("epoch")
    plt.ylabel("eval mean loss")
    plt.title("Evaluation mean loss over epochs")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_ap_per_class(ap_history, num_classes=4, class_names=CLASS_NAMES):
    epochs = range(1, len(ap_history) + 1)
    plt.figure(figsize=(8, 5))
    for c in range(num_classes):
        series = [ep[c] for ep in ap_history]
        plt.plot(epochs, series, marker='o', label=class_names[c])
    plt.xlabel("epoch")
    plt.ylabel("AP@0.5")
    plt.title("AP@0.5 per class over epochs")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_giou_per_class(giou_history, num_classes=4, class_names=CLASS_NAMES):
    epochs = range(1, len(giou_history) + 1)
    plt.figure(figsize=(8, 5))
    for c in range(num_classes):
        series = [ep[c] for ep in giou_history]
        plt.plot(epochs, series, marker='o', label=class_names[c])
    plt.xlabel("epoch")
    plt.ylabel("mean GIoU (matched)")
    plt.title("Mean GIoU per class over epochs")
    plt.ylim(-1, 1)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import cv2
import shutil
import torch
import torch.nn as nn




def save_evaluation_set(eval_set, dataset, save_dir):
    """
    Saves the fixed evaluation set:
      - original images
      - corresponding mask files

    The evaluation set is built once from the original (non-augmented)
    images and remains the same for all epochs during this training run.
    """

    images_dir = os.path.join(save_dir, "images")
    masks_dir = os.path.join(save_dir, "masks")

    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(masks_dir, exist_ok=True)

    saved_ids = set()


    for idx in range(len(eval_set)):


        img_id = dataset.ids[idx]


        img_name = dataset.img_dict[img_id]


        img_src = os.path.join(dataset.img_dir, img_name)


        shutil.copy2(
            img_src,
            os.path.join(images_dir, img_name)
        )


        for mask_name in dataset.mask_dict.get(img_id, []):

            mask_src = os.path.join(
                dataset.mask_dir,
                mask_name
            )

            shutil.copy2(
                mask_src,
                os.path.join(masks_dir, mask_name)
            )

        saved_ids.add(img_id)

    print("\nEvaluation set saved successfully!")
    print(f"  Images: {len(saved_ids)}")
    print(f"  Location: {save_dir}")




train_losses = []
eval_losses = []
train_correct = []
eval_correct = []

model = YOLOv5(num_classes=4)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

criterion = nn.CrossEntropyLoss()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

SAVE_PATH = "/content/drive/MyDrive/fire/model-1/model-1.pth"

os.makedirs(
    os.path.dirname(SAVE_PATH),
    exist_ok=True
)

print(next(model.parameters()).device)

NUM_CLASSES = 4
NO = NUM_CLASSES + 5

ANCHORS_UNITS = [
    a / s
    for a, s in zip(ANCHORS, STRIDES)
]

num_epochs = 30

best_eval_loss = float('inf')
best_mAP      = -1.0

train_losses, eval_losses = [], []
ap_history = []
giou_history = []

start_epoch = 0


'''
if os.path.exists(SAVE_PATH):

    ckpt = torch.load(
        SAVE_PATH,
        map_location=device
    )

    model.load_state_dict(
        ckpt["model"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )

    best_eval_loss = ckpt.get(
        "best_eval_loss",
        float('inf')
    )

    best_mAP = ckpt.get(
        "best_mAP",
        -1.0
    )

    train_losses = ckpt.get(
        "train_losses",
        []
    )

    eval_losses = ckpt.get(
        "eval_losses",
        []
    )

    start_epoch = ckpt.get(
        "epoch",
        -1
    ) + 1

    print(
        f"Loaded checkpoint from {SAVE_PATH} "
        f"| resuming at epoch {start_epoch+1} "
        f"| best_mAP {best_mAP:.4f}"
    )

else:

    print(
        f"No checkpoint found at {SAVE_PATH}; "
        f"training from scratch."
    )
'''




EVAL_SET_PATH = (
    "/content/drive/MyDrive/fire/model-1/"
    "evaluation_set"
)


_eval_imgs = os.path.join(EVAL_SET_PATH, "images")

if not (os.path.isdir(_eval_imgs) and len(os.listdir(_eval_imgs)) > 0):

    save_evaluation_set(
        eval_set,
        eval_set,
        EVAL_SET_PATH
    )

else:

    print(
        f"\nEvaluation set already exists at:"
        f"\n{EVAL_SET_PATH}"
    )




for epoch in range(
    start_epoch,
    start_epoch + num_epochs
):

    print(
        f"\n===== Epoch {epoch+1}/"
        f"{start_epoch + num_epochs} ====="
    )




    model.train()

    train_running = 0.0

    n_train = len(train_loader)

    for i, (inputs, labels) in enumerate(
        train_loader
    ):

        inputs = inputs.to(
            device,
            non_blocking=True
        )

        labels = [
            l.to(device)
            for l in labels
        ]

        head_out = model(inputs)

        loss, parts = compute_detection_loss(
            head_out,
            labels,
            ANCHORS,
            STRIDES,
            GRID_SIZES,
            NA,
            num_classes=NUM_CLASSES
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_running += loss.item()


    train_epoch_loss = (
        train_running / n_train
    )

    train_losses.append(
        train_epoch_loss
    )

    print(
        f"  --> train avg loss: "
        f"{train_epoch_loss:.4f}"
    )




    model.eval()

    eval_running = 0.0

    n_eval = len(eval_loader)

    all_dets, all_gts = [], []


    with torch.no_grad():

        for i, (inputs, labels) in enumerate(
            eval_loader
        ):

            inputs = inputs.to(
                device,
                non_blocking=True
            )

            labels = [
                l.to(device)
                for l in labels
            ]

            head_out = model(inputs)

            loss, parts = compute_detection_loss(
                head_out,
                labels,
                ANCHORS,
                STRIDES,
                GRID_SIZES,
                NA,
                num_classes=NUM_CLASSES
            )

            eval_running += loss.item()




            preds = head_to_nms_input(
                head_out
            )


            for b in range(
                preds.shape[0]
            ):

                dets = non_max_suppression(
                    preds[b],
                    conf_thresh=0.25,
                    iou_thresh=0.45,
                    nc=NUM_CLASSES
                )

                all_dets.append(
                    dets.cpu()
                )




                tgt = labels[b]

                if tgt.shape[0] > 0:

                    tgt_px = tgt.clone()

                    tgt_px[:, 0:4] = (
                        tgt_px[:, 0:4] * 320
                    )

                    gt_xyxy = xywh_to_xyxy(
                        tgt_px[:, 0:4]
                    )

                    gts = torch.cat(
                        [
                            gt_xyxy,
                            tgt_px[:, 4:5]
                        ],
                        dim=1
                    )

                else:

                    gts = torch.zeros(
                        (0, 5)
                    )

                all_gts.append(
                    gts.cpu()
                )


    epoch_eval_loss = (
        eval_running / n_eval
    )

    eval_losses.append(
        epoch_eval_loss
    )


    if epoch_eval_loss < best_eval_loss:

        best_eval_loss = epoch_eval_loss


    print(
        f"  --> eval avg loss: "
        f"{epoch_eval_loss:.4f}"
    )




    ap_per_class, mAP = evaluate_map(
        all_dets,
        all_gts,
        NUM_CLASSES,
        iou_thresh=0.5
    )

    ap_history.append(
        [
            ap_per_class[c]
            for c in range(NUM_CLASSES)
        ]
    )


    print(
        "  --- AP@0.5 per class ---"
    )


    for c in range(NUM_CLASSES):

        ap = ap_per_class[c]

        if ap == ap:

            print(
                f"    class {c}: "
                f"AP = {ap:.4f}"
            )

        else:

            print(
                f"    class {c}: "
                f"AP = n/a (no GT)"
            )


    print(
        f"  --> mAP@0.5: {mAP:.4f}"
    )




    giou_per_class, giou_counts = (
        evaluate_mean_giou(
            all_dets,
            all_gts,
            NUM_CLASSES,
            iou_thresh=0.5
        )
    )

    giou_history.append(
        giou_per_class
    )


    print(
        "  --- mean GIoU per class ---"
    )


    for c in range(NUM_CLASSES):

        print(
            f"    grade{c}: "
            f"{giou_per_class[c]:.4f} "
            f" (n={giou_counts[c]})"
        )




    cm = build_confusion_matrix(
        all_dets,
        all_gts,
        NUM_CLASSES,
        iou_thresh=0.5
    )


    print(
        "  --- confusion matrix "
        "(rows=predicted, cols=ground truth) ---"
    )


    header = (
        "          "
        + "".join(
            [
                f"gt{c:<6}"
                for c in range(NUM_CLASSES)
            ]
        )
    )

    print(header)


    for r in range(NUM_CLASSES):

        row = "".join(
            [
                f"{int(cm[r, col]):<8}"
                for col in range(NUM_CLASSES)
            ]
        )

        print(
            f"    pred{r}  {row}"
        )




    if mAP > best_mAP:

        best_mAP = mAP

        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "best_eval_loss": best_eval_loss,
                "best_mAP": best_mAP,
                "train_losses": train_losses,
                "eval_losses": eval_losses,
                "ap_history": ap_history,
                "giou_history": giou_history,
            },
            SAVE_PATH
        )

        print(
            f"  *** saved new best model "
            f"(mAP {best_mAP:.4f}) "
            f"at epoch {epoch+1} ***"
        )

    else:

        print(
            f"  no improvement "
            f"(best mAP stays {best_mAP:.4f})"
        )


    print(
        f"===== Epoch {epoch+1} done | "
        f"train {train_epoch_loss:.4f} | "
        f"eval {epoch_eval_loss:.4f} | "
        f"mAP {mAP:.4f} ====="
    )




plot_eval_loss(
    eval_losses
)

plot_ap_per_class(
    ap_history,
    NUM_CLASSES
)

plot_giou_per_class(
    giou_history,
    NUM_CLASSES
)